# Walk-Forward Evaluation (Stage 1: Base Rate, XGBoost, LSTM)
FaultLine: replaces the single train/test split with expanding-window
walk-forward evaluation, so model comparisons rest on every M4+ event from 2019
onward instead of the 7 events in the old 2023-2025 test set.

**Fold design** -- each fold trains only on data before its test year:

| Fold | Train | Test | Role |
|---|---|---|---|
| 1 | 2016-2018 | 2019 | tuning |
| 2 | 2016-2019 | 2020 | tuning |
| 3 | 2016-2020 | 2021 | evaluation |
| 4 | 2016-2021 | 2022 | evaluation |
| 5 | 2016-2022 | 2023 | evaluation |
| 6 | 2016-2023 | 2024 | evaluation |
| 7 | 2016-2024 | 2025 | evaluation |

- **Tuning folds** choose settings (XGBoost tree depth, LSTM epoch count).
- **Evaluation folds** run once with those settings locked in. Only these are
  reported, so the final numbers were never used to make a decision.
- A 30-day **embargo** before each test year keeps training labels from peeking
  into the test year.

**Three models:**
1. **Base rate** -- each section's historical M4+ frequency from the training
   years; no features. The bar the ML models must clear to show the dynamic
   features add anything.
2. **XGBoost** -- same features/settings as `04_xgboost_model.ipynb`.
3. **LSTM** -- same architecture as `05_lstm_model.ipynb`, but a fixed epoch
   count chosen on the tuning folds instead of early stopping on a 2-event
   validation year.

**Two evaluation grids:**
- **Monthly (primary):** forecasts issued on the 1st of each month. Adjacent
  30-day windows barely overlap, so each forecast is close to an independent
  data point -- and it's the grid ETAS will be evaluated on in Stage 2.
- **Daily (secondary):** every day. More rows, but consecutive days share 29 of
  30 label days, so it overstates how much evidence there really is.


In [1]:
import torch                      # must be imported FIRST on Windows (PyTorch 2.9+ DLL-order bug)
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import xgboost as xgb
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)


## Configuration

In [2]:
MAG_THRESHOLD = 3.5
FORECAST_DAYS = 30
PROJECTED_CRS = "EPSG:3310"
MAX_ASSIGN_DIST_KM = 20

TUNING_YEARS = [2019, 2020]
EVAL_YEARS = [2021, 2022, 2023, 2024, 2025]

# XGBoost: fixed settings from 04_xgboost_model.ipynb, except tree depth, which is tuned
XGB_BASE_PARAMS = {"n_estimators": 300, "learning_rate": 0.05, "eval_metric": "logloss"}
XGB_DEPTH_CANDIDATES = [3, 4, 6]

# LSTM: fixed architecture from 05_lstm_model.ipynb; epoch count is tuned
SEQ_LEN = 60
HIDDEN_SIZE = 64
DROPOUT = 0.2
BATCH_SIZE = 256
LEARNING_RATE = 1e-3
LSTM_EPOCH_CANDIDATES = [5, 10, 20]   # one 20-epoch run per tuning fold, scored at each checkpoint

NUMERIC_COLS = [
    "seismicity_rate_7d", "seismicity_rate_30d", "seismicity_rate_90d", "seismicity_rate_30d_z",
    "b_value", "b_value_se", "n_events", "gps_strain_rate", "gps_station_distance_km",
    "max_mag_7d", "max_mag_30d", "days_since_last_m35",
]
BOOL_COLS = ["b_value_reliable", "gps_reliable"]
XGB_FEATURE_COLS = ["fault_segment_id"] + NUMERIC_COLS + BOOL_COLS


## Load features and build labels

Same guards and label logic as the model notebooks. Labels are built once for
every day; each fold just selects different date ranges.

In [3]:
features = pd.read_parquet("../data/processed/features.parquet")
features["date"] = pd.to_datetime(features["date"])
if features["date"].dt.tz is not None:
    features["date"] = features["date"].dt.tz_localize(None)
assert features["gps_strain_rate"].std() > 0, (
    "gps_strain_rate has zero variance -- features.parquet is stale. "
    "Restart & Run All in 02_feature_engineering.ipynb first."
)

missing = {"max_mag_7d", "max_mag_30d", "days_since_last_m35"} - set(features.columns)
assert not missing, f"features.parquet is missing {missing} -- rerun 02_feature_engineering.ipynb"

faults = gpd.read_file("../data/raw/san_andreas_faults.geojson")
faults = faults.dissolve(by="section_na").reset_index()
faults["fault_segment_id"] = faults.index.astype(str)

quakes = pd.read_parquet("../data/raw/usgs_2016_2026.parquet")
quakes["time"] = pd.to_datetime(quakes["time"])
if quakes["time"].dt.tz is not None:
    quakes["time"] = quakes["time"].dt.tz_localize(None)
# NOTE: check quakes.columns.tolist() and adjust if your schema differs
quakes = quakes.rename(columns={"event_id": "id", "mag": "magnitude"})

quake_points_m = gpd.GeoDataFrame(
    quakes.copy(), geometry=gpd.points_from_xy(quakes["longitude"], quakes["latitude"]), crs="EPSG:4326"
).to_crs(PROJECTED_CRS)
joined = gpd.sjoin_nearest(
    quake_points_m, faults.to_crs(PROJECTED_CRS)[["fault_segment_id", "geometry"]],
    max_distance=MAX_ASSIGN_DIST_KM * 1000, distance_col="dist_to_fault_m",
)
quakes_assigned = joined.drop(columns="geometry").rename(columns={"time": "event_time"})


def build_labels(features_dates, quakes_assigned, mag_threshold, forecast_days):
    qualifying = quakes_assigned[quakes_assigned["magnitude"] >= mag_threshold].copy()
    qualifying["event_date"] = qualifying["event_time"].dt.floor("D")
    labels = []
    for seg_id, seg_dates in features_dates.groupby("fault_segment_id"):
        seg_events = qualifying[qualifying["fault_segment_id"] == seg_id]["event_date"].sort_values().values
        for d in seg_dates["date"]:
            window_end = d + pd.Timedelta(days=forecast_days)
            has_event = np.any((seg_events > np.datetime64(d)) & (seg_events <= np.datetime64(window_end)))
            labels.append({"fault_segment_id": seg_id, "date": d, "label": int(has_event)})
    return pd.DataFrame(labels)


labels = build_labels(features[["fault_segment_id", "date"]], quakes_assigned, MAG_THRESHOLD, FORECAST_DAYS)
data = features.merge(labels, on=["fault_segment_id", "date"], how="left")
data["fault_segment_id"] = data["fault_segment_id"].astype(str)
SEGMENT_IDS = sorted(data["fault_segment_id"].unique(), key=int)
DATA_END = data["date"].max()

qa = quakes_assigned[quakes_assigned["magnitude"] >= MAG_THRESHOLD]
print("Distinct M4+ events per year (all sections):")
print(qa.groupby(qa["event_time"].dt.year).size())


Distinct M4+ events per year (all sections):
event_time
2016    17
2017    10
2018    10
2019    11
2020    14
2021    22
2022     5
2023    10
2024     4
2025     9
dtype: int64


## Fold masks -- the one piece that must never leak

For test year `Y`:
- **Train:** dates before Jan 1 of `Y`, minus a 30-day embargo, so no training
  label's 30-day window reaches into the test year.
- **Test (daily):** dates in `Y` whose 30-day window is fully covered by the
  catalog (only matters for the final year).
- **Test (monthly):** the subset of test dates that fall on the 1st of a month.

The asserts check the embargo on every fold, every time the notebook runs.

In [4]:
def fold_masks(dates, test_year):
    test_start = pd.Timestamp(f"{test_year}-01-01")
    test_end = pd.Timestamp(f"{test_year}-12-31")
    train_cutoff = test_start - pd.Timedelta(days=FORECAST_DAYS)
    train = dates < train_cutoff
    test_daily = (dates >= test_start) & (dates <= test_end) & (dates + pd.Timedelta(days=FORECAST_DAYS) <= DATA_END)
    test_monthly = test_daily & (dates.dt.day == 1)

    # no training label's window may reach the test year
    assert (dates[train] + pd.Timedelta(days=FORECAST_DAYS) < test_start).all(), "embargo violated"
    assert not (train & test_daily).any(), "train/test overlap"
    return train, test_daily, test_monthly


for y in TUNING_YEARS + EVAL_YEARS:
    tr, td, tm = fold_masks(data["date"], y)
    print(f"Test {y}: train rows {tr.sum():>6}, test daily {td.sum():>5}, test monthly {tm.sum():>4}, "
          f"monthly positives {int(data.loc[tm, 'label'].sum())}")


Test 2019: train rows  10650, test daily  3650, test monthly  120, monthly positives 8
Test 2020: train rows  14300, test daily  3660, test monthly  120, monthly positives 8
Test 2021: train rows  17960, test daily  3650, test monthly  120, monthly positives 12
Test 2022: train rows  21610, test daily  3650, test monthly  120, monthly positives 4
Test 2023: train rows  25260, test daily  3650, test monthly  120, monthly positives 8
Test 2024: train rows  28910, test daily  3660, test monthly  120, monthly positives 3
Test 2025: train rows  32570, test daily  3350, test monthly  120, monthly positives 5


## Shared helpers

**Undoing class weighting.** XGBoost's `scale_pos_weight` and the LSTM's
`pos_weight` inflate raw probabilities (a weight of ~55 turns a realistic 7%
into ~80%). `deweight()` divides the odds back down so Brier scores are
comparable with the base-rate model. Ranking metrics (PR-AUC, ROC-AUC) are
unaffected, since the correction never changes the order of predictions. This is
an approximate correction; proper calibration is a later step.

**Scoring.** A fold with no positives has no defined PR-AUC/ROC-AUC, so those
come back as NaN instead of crashing.

In [5]:
def deweight(probs, weight):
    probs = np.clip(probs, 1e-7, 1 - 1e-7)
    odds = probs / (1 - probs) / weight
    return odds / (1 + odds)


def score(y, p):
    y = np.asarray(y)
    n_pos = int(y.sum())
    return {
        "rows": len(y),
        "positives": n_pos,
        "pr_auc": average_precision_score(y, p) if n_pos > 0 else np.nan,
        "roc_auc": roc_auc_score(y, p) if 0 < n_pos < len(y) else np.nan,
        "brier": brier_score_loss(y, p),
    }


## Model 1: historical base rate

Each section's fraction of positive training days. No features, no learning --
just "how often has this section had an M4+ within 30 days, historically?"

In [6]:
def predict_base_rate(train_df, test_df):
    rates = train_df.groupby("fault_segment_id")["label"].mean()
    overall = train_df["label"].mean()
    return test_df["fault_segment_id"].map(rates).fillna(overall).to_numpy()


## Model 2: XGBoost

`fault_segment_id` uses a fixed category list on every fold so the model always
sees the same 10 categories, even if a section happens to be absent from one
fold's rows.

In [7]:
def xgb_frame(df, feature_cols=XGB_FEATURE_COLS):
    X = df[feature_cols].copy()                                   # FIX 1
    X["fault_segment_id"] = pd.Categorical(X["fault_segment_id"], categories=SEGMENT_IDS)
    for col in BOOL_COLS:
        if col in X:                                              # FIX 2
            X[col] = X[col].astype(float)
    return X


def train_predict_xgb(train_df, test_df, max_depth, feature_cols=XGB_FEATURE_COLS):
    y = train_df["label"].to_numpy()
    weight = (y == 0).sum() / max((y == 1).sum(), 1)
    model = xgb.XGBClassifier(enable_categorical=True, scale_pos_weight=weight,
                              max_depth=max_depth, random_state=SEED, **XGB_BASE_PARAMS)
    model.fit(xgb_frame(train_df, feature_cols), y)                    # FIX 3
    raw = model.predict_proba(xgb_frame(test_df, feature_cols))[:, 1]  # FIX 4
    return deweight(raw, weight), model

## Model 3: LSTM

Preprocessing (median fill, missing flags, z-scoring) is re-fit on **each
fold's training rows**, then sequences are rebuilt -- a fold's scaler must never
see its own test year. Sequences are built on the full unbroken timeline (so
windows never stitch across a gap), then selected by date.

In [8]:
def fit_preprocessor(train_df, numeric_cols):
    params = {}
    for col in numeric_cols:
        s = train_df[col].astype(float)
        median = float(s.median()) if s.notna().any() else 0.0
        filled = s.fillna(median)
        std = float(filled.std())
        params[col] = {"median": median, "mean": float(filled.mean()),
                       "std": std if std > 0 else 1.0, "add_missing_flag": bool(s.isna().any())}
    return params


def apply_preprocessor(df, params):
    out = df.copy()
    for col, p in params.items():
        if p["add_missing_flag"]:
            out[f"{col}_missing"] = out[col].isna().astype(np.float32)
        out[col] = (out[col].astype(float).fillna(p["median"]) - p["mean"]) / p["std"]
    return out


def build_sequences(df, feature_cols, seq_len):
    X_parts, meta_parts = [], []
    for seg_id, seg in df.groupby("fault_segment_id"):
        seg = seg.sort_values("date")
        assert (seg["date"].diff().dropna().dt.days == 1).all(), f"segment {seg_id} has non-contiguous dates"
        F = seg[feature_cols].to_numpy(dtype=np.float32)
        if len(F) < seq_len:
            continue
        windows = np.lib.stride_tricks.sliding_window_view(F, seq_len, axis=0)
        X_parts.append(windows.transpose(0, 2, 1))
        meta_parts.append(seg.iloc[seq_len - 1:][["fault_segment_id", "date", "label"]])
    return np.ascontiguousarray(np.concatenate(X_parts)), pd.concat(meta_parts, ignore_index=True)


def lstm_fold_sequences(data, train_cutoff, numeric_cols=NUMERIC_COLS, bool_cols=BOOL_COLS):
    params = fit_preprocessor(data[data["date"] < train_cutoff], numeric_cols)
    prep = apply_preprocessor(data, params)
    for col in bool_cols:
        prep[col] = prep[col].astype(np.float32)
    onehot = pd.get_dummies(prep["fault_segment_id"], prefix="seg").reindex(
        columns=[f"seg_{s}" for s in SEGMENT_IDS], fill_value=0).astype(np.float32)
    prep = pd.concat([prep, onehot], axis=1)
    flags = [f"{c}_missing" for c, p in params.items() if p["add_missing_flag"]]
    input_cols = numeric_cols + flags + bool_cols + list(onehot.columns)
    assert not prep[input_cols].isna().any().any(), "NaN survived preprocessing"
    return build_sequences(prep, input_cols, SEQ_LEN)


class FaultLineLSTM(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.lstm = nn.LSTM(n_features, HIDDEN_SIZE, batch_first=True)
        self.dropout = nn.Dropout(DROPOUT)
        self.head = nn.Linear(HIDDEN_SIZE, 1)

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        return self.head(self.dropout(h_n[-1])).squeeze(-1)


def predict_proba(model, X, batch_size=1024):
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            out.append(torch.sigmoid(model(torch.from_numpy(X[i:i + batch_size]))).numpy())
    return np.concatenate(out)


def train_predict_lstm(X_train, y_train, X_test, checkpoints):
    """Train for max(checkpoints) epochs; return de-weighted test predictions
    at each checkpoint epoch (so one run can score several epoch counts)."""
    torch.manual_seed(SEED)
    model = FaultLineLSTM(X_train.shape[2])
    weight = float((y_train == 0).sum() / max((y_train == 1).sum(), 1))
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(weight, dtype=torch.float32))
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    loader = DataLoader(TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train)),
                        batch_size=BATCH_SIZE, shuffle=True, generator=torch.Generator().manual_seed(SEED))
    preds = {}
    for epoch in range(1, max(checkpoints) + 1):
        model.train()
        for xb, yb in loader:
            optimizer.zero_grad()
            loss_fn(model(xb), yb).backward()
            optimizer.step()
        if epoch in checkpoints:
            preds[epoch] = deweight(predict_proba(model, X_test), weight)
    return preds


def lstm_fold(test_year, checkpoints, numeric_cols=NUMERIC_COLS, bool_cols=BOOL_COLS):
    test_start = pd.Timestamp(f"{test_year}-01-01")
    X_all, meta = lstm_fold_sequences(data, test_start - pd.Timedelta(days=FORECAST_DAYS),
                                      numeric_cols, bool_cols)
    tr, td, tm = fold_masks(meta["date"], test_year)
    y_all = meta["label"].to_numpy(dtype=np.float32)
    preds = train_predict_lstm(X_all[tr.to_numpy()], y_all[tr.to_numpy()], X_all[td.to_numpy()], checkpoints)
    return preds, meta[td].reset_index(drop=True)


## Tuning folds (2019, 2020): choose XGBoost depth and LSTM epochs

Scored on **pooled daily PR-AUC** across both tuning years. Daily rather than
monthly here because the monthly grid over only two years has very few
positives -- too few to reliably tell settings apart. This is fine: these years
are only for choosing settings and are never reported as results.

In [9]:
tuning_rows = []
for y in TUNING_YEARS:
    tr, td, _ = fold_masks(data["date"], y)
    train_df, test_df = data[tr], data[td]
    for depth in XGB_DEPTH_CANDIDATES:
        p, _ = train_predict_xgb(train_df, test_df, depth)
        tuning_rows.append(pd.DataFrame({"model": "xgboost", "setting": depth, "label": test_df["label"].to_numpy(), "prob": p}))
    lstm_preds, lstm_meta = lstm_fold(y, LSTM_EPOCH_CANDIDATES)
    for epochs, p in lstm_preds.items():
        tuning_rows.append(pd.DataFrame({"model": "lstm", "setting": epochs, "label": lstm_meta["label"].to_numpy(), "prob": p}))
    print(f"Tuning fold {y} done")

tuning = pd.concat(tuning_rows, ignore_index=True)
tuning_scores = tuning.groupby(["model", "setting"]).apply(
    lambda g: pd.Series(score(g["label"], g["prob"]))).reset_index()
print(tuning_scores)

best_depth = int(tuning_scores[tuning_scores["model"] == "xgboost"].sort_values("pr_auc", ascending=False).iloc[0]["setting"])
best_epochs = int(tuning_scores[tuning_scores["model"] == "lstm"].sort_values("pr_auc", ascending=False).iloc[0]["setting"])
print(f"\nLocked in: XGBoost max_depth={best_depth}, LSTM epochs={best_epochs}")


Tuning fold 2019 done
Tuning fold 2020 done
     model  setting    rows  positives    pr_auc   roc_auc     brier
0     lstm        5  7310.0      541.0  0.227408  0.708050  0.069340
1     lstm       10  7310.0      541.0  0.184017  0.661766  0.069821
2     lstm       20  7310.0      541.0  0.168138  0.644196  0.076045
3  xgboost        3  7310.0      541.0  0.158673  0.674474  0.070895
4  xgboost        4  7310.0      541.0  0.124824  0.648346  0.074539
5  xgboost        6  7310.0      541.0  0.125180  0.667934  0.080400

Locked in: XGBoost max_depth=3, LSTM epochs=5


C:\Users\matth\AppData\Local\Temp\ipykernel_19008\3843994884.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  tuning_scores = tuning.groupby(["model", "setting"]).apply(


## Evaluation folds (2021-2025): run once with settings locked

Every prediction is stored with its date, section, and label, so metrics can be
sliced any way afterwards -- and so ETAS predictions can be merged in on the same
`(fault_segment_id, date)` keys in Stage 2.

In [10]:
NO_GPS_COLS = [c for c in XGB_FEATURE_COLS
               if c not in ("gps_strain_rate", "gps_station_distance_km", "gps_reliable")]
NO_GPS_NUMERIC = [c for c in NUMERIC_COLS if c not in ("gps_strain_rate", "gps_station_distance_km")]
NO_GPS_BOOL = [c for c in BOOL_COLS if c != "gps_reliable"]
lstm_ng, lstm_ng_meta = lstm_fold(y, [best_epochs], NO_GPS_NUMERIC, NO_GPS_BOOL)

pred_rows = []
for y in EVAL_YEARS:
    tr, td, _ = fold_masks(data["date"], y)
    train_df, test_df = data[tr], data[td]
    keys = test_df[["fault_segment_id", "date", "label"]].reset_index(drop=True)

    pred_rows.append(keys.assign(model="base_rate", fold=y, prob=predict_base_rate(train_df, test_df)))
    xgb_p, _ = train_predict_xgb(train_df, test_df, best_depth)
    pred_rows.append(keys.assign(model="xgboost", fold=y, prob=xgb_p))

    xgb_no_gps, _ = train_predict_xgb(train_df, test_df, best_depth, NO_GPS_COLS)
    pred_rows.append(keys.assign(model="xgboost_no_gps", fold=y, prob=xgb_no_gps))

    lstm_preds, lstm_meta = lstm_fold(y, [best_epochs])
    pred_rows.append(lstm_meta.assign(model="lstm", fold=y, prob=lstm_preds[best_epochs]))

    lstm_ng, lstm_ng_meta = lstm_fold(y, [best_epochs], NO_GPS_NUMERIC, NO_GPS_BOOL)
    pred_rows.append(lstm_ng_meta.assign(model="lstm_no_gps", fold=y, prob=lstm_ng[best_epochs]))

    print(f"Evaluation fold {y} done")

predictions = pd.concat(pred_rows, ignore_index=True)
predictions["monthly"] = predictions["date"].dt.day == 1

# every model must be scored on exactly the same rows, or the comparison isn't fair
counts = predictions.groupby(["fold", "model"]).size().unstack()
assert (counts.nunique(axis=1) == 1).all(), f"models scored on different rows:\n{counts}"


Evaluation fold 2021 done
Evaluation fold 2022 done
Evaluation fold 2023 done
Evaluation fold 2024 done
Evaluation fold 2025 done


## Results

**Pooled** = all evaluation years stacked together (the headline numbers).
**Per-fold** = each year separately (how much performance varies year to year).

How to read them:
- **PR-AUC** -- compare against the positive rate (what a random guess scores).
  The key question is whether XGBoost/LSTM beat **base_rate**.
- **ROC-AUC** -- 0.5 is random ordering.
- **Brier** -- lower is better; compares probability accuracy directly, possible
  now that the class-weighting inflation has been undone.

In [11]:
m = predictions[predictions["monthly"]].copy()

# 1. Does XGBoost's average prediction drift from year to year, independent of actual activity?
print(m.groupby(["fold", "model"])["prob"].mean().unstack().round(4)
        .join(m.groupby("fold")["label"].mean().rename("actual_rate")))

# 2. Remove the year-to-year level differences: convert each model's predictions to
#    percentile ranks WITHIN each year, then pool. Only PR-AUC and ROC-AUC are
#    meaningful here (ranks aren't probabilities, so ignore Brier).
m["prob_rank"] = m.groupby(["fold", "model"])["prob"].rank(pct=True)
print(m.groupby("model").apply(lambda g: pd.Series(score(g["label"], g["prob_rank"])),
                               include_groups=False).round(4))

      base_rate    lstm  lstm_no_gps  xgboost  xgboost_no_gps  actual_rate
fold                                                                      
2021     0.0703  0.0614       0.0735   0.0405          0.0477     0.100000
2022     0.0757  0.0669       0.0865   0.0522          0.0499     0.033333
2023     0.0709  0.0500       0.0899   0.0665          0.0669     0.066667
2024     0.0712  0.0889       0.0845   0.0682          0.0452     0.025000
2025     0.0660  0.1335       0.0626   0.0264          0.0546     0.041667
                 rows  positives  pr_auc  roc_auc   brier
model                                                    
base_rate       600.0       32.0  0.1259   0.7360  0.3112
lstm            600.0       32.0  0.1330   0.6992  0.3170
lstm_no_gps     600.0       32.0  0.1198   0.7069  0.3162
xgboost         600.0       32.0  0.1273   0.6987  0.3170
xgboost_no_gps  600.0       32.0  0.1157   0.6916  0.3177


In [12]:
def results_table(df, by):
    return df.groupby(by).apply(lambda g: pd.Series(score(g["label"], g["prob"]))).round(4)

monthly = predictions[predictions["monthly"]]
print("=== POOLED, MONTHLY GRID (primary) ===")
print(results_table(monthly, ["model"]))
print("\n=== POOLED, DAILY GRID (secondary) ===")
print(results_table(predictions, ["model"]))


=== POOLED, MONTHLY GRID (primary) ===
                 rows  positives  pr_auc  roc_auc   brier
model                                                    
base_rate       600.0       32.0  0.1455   0.7406  0.0497
lstm            600.0       32.0  0.1162   0.6772  0.0599
lstm_no_gps     600.0       32.0  0.1177   0.6987  0.0575
xgboost         600.0       32.0  0.1316   0.6921  0.0534
xgboost_no_gps  600.0       32.0  0.1256   0.6974  0.0530

=== POOLED, DAILY GRID (secondary) ===


C:\Users\matth\AppData\Local\Temp\ipykernel_19008\3809574793.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby(by).apply(lambda g: pd.Series(score(g["label"], g["prob"]))).round(4)


                   rows  positives  pr_auc  roc_auc   brier
model                                                      
base_rate       17960.0     1022.0  0.1678   0.7573  0.0513
lstm            17960.0     1022.0  0.1455   0.7041  0.0593
lstm_no_gps     17960.0     1022.0  0.1242   0.7137  0.0590
xgboost         17960.0     1022.0  0.1269   0.7166  0.0561
xgboost_no_gps  17960.0     1022.0  0.1212   0.7153  0.0557


C:\Users\matth\AppData\Local\Temp\ipykernel_19008\3809574793.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby(by).apply(lambda g: pd.Series(score(g["label"], g["prob"]))).round(4)


In [13]:
print("=== PER-FOLD, MONTHLY GRID: PR-AUC ===")
per_fold = results_table(monthly, ["fold", "model"]).reset_index()
print(per_fold.pivot(index="fold", columns="model", values="pr_auc"))
print("\nPositives per fold (monthly):")
print(per_fold.drop_duplicates("fold").set_index("fold")["positives"])

print("\n=== PER-FOLD, DAILY GRID: PR-AUC ===")
print(results_table(predictions, ["fold", "model"]).reset_index().pivot(index="fold", columns="model", values="pr_auc"))


=== PER-FOLD, MONTHLY GRID: PR-AUC ===


C:\Users\matth\AppData\Local\Temp\ipykernel_19008\3809574793.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby(by).apply(lambda g: pd.Series(score(g["label"], g["prob"]))).round(4)


model  base_rate    lstm  lstm_no_gps  xgboost  xgboost_no_gps
fold                                                          
2021      0.3028  0.4314       0.3474   0.4711          0.3781
2022      0.0833  0.1008       0.0752   0.0538          0.0969
2023      0.1121  0.0860       0.0926   0.2186          0.1523
2024      0.0764  0.2253       0.1406   0.0804          0.0727
2025      0.0917  0.1586       0.1333   0.0919          0.0680

Positives per fold (monthly):
fold
2021    12.0
2022     4.0
2023     8.0
2024     3.0
2025     5.0
Name: positives, dtype: float64

=== PER-FOLD, DAILY GRID: PR-AUC ===
model  base_rate    lstm  lstm_no_gps  xgboost  xgboost_no_gps
fold                                                          
2021      0.2294  0.3032       0.3070   0.2826          0.2395
2022      0.1914  0.1910       0.1325   0.1354          0.1212
2023      0.1226  0.1287       0.0828   0.1523          0.1211
2024      0.0708  0.0934       0.0925   0.1272          0.0671
2025      

C:\Users\matth\AppData\Local\Temp\ipykernel_19008\3809574793.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby(by).apply(lambda g: pd.Series(score(g["label"], g["prob"]))).round(4)


## Per-section view across all evaluation years

Same check that exposed the non-stationarity problem: each section's actual rate
vs. each model's average prediction, pooled across 2021-2025.

In [14]:
section_view = predictions.groupby(["fault_segment_id", "model"]).agg(
    actual_rate=("label", "mean"), mean_predicted=("prob", "mean")).reset_index()
section_view = section_view.pivot(index="fault_segment_id", columns="model", values="mean_predicted").join(
    predictions.groupby("fault_segment_id")["label"].mean().rename("actual_rate"))
section_view = section_view.merge(faults[["fault_segment_id", "section_na"]], left_index=True,
                                  right_on="fault_segment_id").set_index("section_na")
print(section_view.drop(columns="fault_segment_id").sort_values("actual_rate", ascending=False).round(4))


                                  base_rate    lstm  lstm_no_gps  xgboost  xgboost_no_gps  actual_rate
section_na                                                                                            
Creeping section                     0.2801  0.2780       0.3305   0.2245          0.2210       0.1943
San Bernardino Mountains section     0.1597  0.1524       0.1675   0.0829          0.1139       0.1281
Santa Cruz Mountains section         0.0673  0.0870       0.0321   0.0414          0.0393       0.0930
Coachella section                    0.1047  0.1077       0.1531   0.0820          0.0741       0.0702
Parkfield section                    0.0288  0.0367       0.0252   0.0151          0.0174       0.0167
Mojave section                       0.0000  0.0155       0.0009   0.0015          0.0003       0.0167
Peninsula section                    0.0163  0.0224       0.0294   0.0050          0.0048       0.0167
North Coast section                  0.0038  0.0284       0.0040   0.0027

In [15]:
m = predictions[predictions["monthly"]].copy()
m["prob_rank"] = m.groupby(["fold", "model"])["prob"].rank(pct=True)
print("=== POOLED, MONTHLY, WITHIN-YEAR RANKS (PR-AUC / ROC-AUC only) ===")
print(m.groupby("model").apply(lambda g: pd.Series(score(g["label"], g["prob_rank"])),
                               include_groups=False)[["positives", "pr_auc", "roc_auc"]].round(4))

=== POOLED, MONTHLY, WITHIN-YEAR RANKS (PR-AUC / ROC-AUC only) ===
                positives  pr_auc  roc_auc
model                                     
base_rate            32.0  0.1259   0.7360
lstm                 32.0  0.1330   0.6992
lstm_no_gps          32.0  0.1198   0.7069
xgboost              32.0  0.1273   0.6987
xgboost_no_gps       32.0  0.1157   0.6916


In [16]:
m = predictions[predictions["monthly"]].merge(
    data[["fault_segment_id", "date", "days_since_last_m35"]], on=["fault_segment_id", "date"])
m["recent_m35"] = m["days_since_last_m35"] <= 30

summary = m.groupby(["recent_m35", "model"]).agg(
    forecasts=("label", "size"), actual_rate=("label", "mean"), mean_predicted=("prob", "mean")
).round(4)
print(summary)

                           forecasts  actual_rate  mean_predicted
recent_m35 model                                                 
False      base_rate             562       0.0498          0.0645
           lstm                  562       0.0498          0.0719
           lstm_no_gps           562       0.0498          0.0697
           xgboost               562       0.0498          0.0431
           xgboost_no_gps        562       0.0498          0.0430
True       base_rate              38       0.1053          0.1637
           lstm                   38       0.1053          0.2024
           lstm_no_gps            38       0.1053          0.2226
           xgboost                38       0.1053          0.1642
           xgboost_no_gps         38       0.1053          0.1988


In [17]:
_, model_full = train_predict_xgb(train_df, test_df, best_depth)
_, model_no_gps = train_predict_xgb(train_df, test_df, best_depth, NO_GPS_COLS)

print("Full model features:  ", model_full.get_booster().feature_names)
print("No-GPS model features:", model_no_gps.get_booster().feature_names)

gain = model_full.get_booster().get_score(importance_type="gain")
print("\nHow much the full model actually used each GPS feature (0 = never split on):")
for col in ["gps_strain_rate", "gps_station_distance_km", "gps_reliable"]:
    print(f"  {col}: {gain.get(col, 0):.3f}")
print("\nTop features by gain:")
print(sorted(gain.items(), key=lambda kv: -kv[1])[:8])

Full model features:   ['fault_segment_id', 'seismicity_rate_7d', 'seismicity_rate_30d', 'seismicity_rate_90d', 'seismicity_rate_30d_z', 'b_value', 'b_value_se', 'n_events', 'gps_strain_rate', 'gps_station_distance_km', 'max_mag_7d', 'max_mag_30d', 'days_since_last_m35', 'b_value_reliable', 'gps_reliable']
No-GPS model features: ['fault_segment_id', 'seismicity_rate_7d', 'seismicity_rate_30d', 'seismicity_rate_90d', 'seismicity_rate_30d_z', 'b_value', 'b_value_se', 'n_events', 'max_mag_7d', 'max_mag_30d', 'days_since_last_m35', 'b_value_reliable']

How much the full model actually used each GPS feature (0 = never split on):
  gps_strain_rate: 174.925
  gps_station_distance_km: 0.000
  gps_reliable: 0.000

Top features by gain:
[('fault_segment_id', 665.9725341796875), ('seismicity_rate_30d', 527.4627685546875), ('seismicity_rate_90d', 341.0938720703125), ('n_events', 241.09033203125), ('days_since_last_m35', 236.4285888671875), ('max_mag_30d', 175.39512634277344), ('gps_strain_rate', 1

In [18]:
os.makedirs("../data/processed", exist_ok=True)
predictions.to_parquet("../data/processed/walk_forward_predictions_m35_strainrate_ablation.parquet", index=False)
results_table(monthly, ["fold", "model"]).to_csv("../data/processed/walk_forward_metrics_monthly.csv")
print("Saved walk_forward_predictions_m35.parquet and walk_forward_metrics_monthly.csv")


Saved walk_forward_predictions_m35.parquet and walk_forward_metrics_monthly.csv


C:\Users\matth\AppData\Local\Temp\ipykernel_19008\3809574793.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby(by).apply(lambda g: pd.Series(score(g["label"], g["prob"]))).round(4)
